# EXPERIMENT 5B – Boosting and Voting Classifiers on Iris Dataset

**Aim:** Apply AdaBoost, Gradient Tree Boosting (GBM), XGBoost, Hard Voting and Soft Voting on the Iris dataset and compare their performance.

**Dataset:** [Kaggle Iris](https://www.kaggle.com/datasets/uciml/iris)

**Evaluation:** Accuracy, macro precision, macro recall, macro F1-score and confusion matrix.

## 1. Install and import libraries
XGBoost is an additional package. If it is not installed, uncomment and run the pip command once.

In [ ]:
# In Jupyter, uncomment this line if XGBoost is not installed:
# %pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import (
    AdaBoostClassifier, GradientBoostingClassifier,
    RandomForestClassifier, VotingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

print("Libraries imported successfully.")

## 2. Load Iris dataset
Download `Iris.csv` from Kaggle and keep it in the same folder as this notebook.

In [ ]:
iris = pd.read_csv("Iris.csv")
display(iris.head())
print("Dataset shape:", iris.shape)
print("Columns:", iris.columns.tolist())
print("\nMissing values:\n", iris.isnull().sum())

## 3. Prepare features and target
The `Id` column is an identifier and is removed. The four flower measurements are used as features. LabelEncoder converts species names to integer labels for XGBoost.

In [ ]:
X = iris.drop(columns=["Id", "Species"], errors="ignore")
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(iris["Species"])

print("Species encoding:")
for i, name in enumerate(label_encoder.classes_):
    print(name, "=", i)
print("Features:", X.columns.tolist())

## 4. Split the dataset
Use 80% for training and 20% for testing. Stratification preserves the proportion of each species.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 5. Create ensemble models

- **AdaBoost:** Trains weak learners sequentially and gives more attention to previously misclassified samples.
- **Gradient Boosting (GBM):** Adds trees sequentially to reduce the errors of the current ensemble.
- **XGBoost:** An optimized gradient boosting implementation.
- **Hard Voting:** Selects the class receiving the most votes.
- **Soft Voting:** Averages predicted class probabilities and selects the highest average probability.

In [ ]:
# AdaBoost classifier
ada_model = AdaBoostClassifier(
    n_estimators=50, learning_rate=1.0, random_state=42
)

# Gradient Tree Boosting classifier
gbm_model = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
)

# XGBoost classifier for the three Iris species
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

# Base models for voting; scaling is used for Logistic Regression and SVM
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
svm_model = make_pipeline(
    StandardScaler(),
    SVC(probability=True, random_state=42)
)

# Hard voting combines class-label votes
hard_voting_model = VotingClassifier(
    estimators=[("lr", logistic_model), ("rf", forest_model), ("svm", svm_model)],
    voting="hard"
)

# Soft voting combines predicted class probabilities
soft_voting_model = VotingClassifier(
    estimators=[("lr", logistic_model), ("rf", forest_model), ("svm", svm_model)],
    voting="soft"
)

models = {
    "AdaBoost": ada_model,
    "Gradient Boosting": gbm_model,
    "XGBoost": xgb_model,
    "Hard Voting": hard_voting_model,
    "Soft Voting": soft_voting_model
}
print("Models created.")

## 6. Train all models
All models use the same training data and test split for a fair comparison.

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "training completed.")

## 7. Generate predictions

In [ ]:
predictions = {}
for name, model in models.items():
    predictions[name] = model.predict(X_test)

print("Predictions generated for all models.")

## 8. Compare evaluation measures

- **Accuracy:** Overall proportion of correct predictions.
- **Macro precision:** Precision averaged equally across the three species.
- **Macro recall:** Recall averaged equally across the three species.
- **Macro F1-score:** F1-score averaged equally across the three species.

Macro averaging gives each species equal weight.

In [ ]:
results = []

for name, y_pred in predictions.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

## 9. Plot performance comparison

In [ ]:
plot_data = results_df.set_index("Model")
plot_data[["Accuracy", "Precision", "Recall", "F1 Score"]].plot(
    kind="bar", figsize=(11, 5)
)
plt.title("Performance Comparison of Ensemble Models")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

## 10. Confusion matrices
The confusion matrix compares actual species with predicted species. The diagonal represents correct predictions.

In [ ]:
class_names = label_encoder.classes_

for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )
    disp.plot(cmap="Blues", values_format="d")
    plt.title(name + " – Confusion Matrix")
    plt.xticks(rotation=20)
    plt.show()

## 11. Classification reports
This displays precision, recall and F1-score separately for each Iris species.

In [ ]:
for name, y_pred in predictions.items():
    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)
    print(classification_report(
        y_test, y_pred, target_names=class_names, zero_division=0
    ))

## 12. Identify the best model based on accuracy

In [ ]:
best = results_df.loc[results_df["Accuracy"].idxmax()]
print("Best model:", best["Model"])
print("Test accuracy:", round(best["Accuracy"] * 100, 2), "%")
display(results_df.sort_values("Accuracy", ascending=False).round(4))

## 13. Conclusion

AdaBoost, Gradient Tree Boosting, XGBoost, Hard Voting and Soft Voting were implemented on the Iris dataset. The models were compared using accuracy, macro precision, macro recall, macro F1-score and confusion matrices. The model with the highest test accuracy is identified in the output.

**Result:** The comparison table, graphs and confusion matrices are displayed after running the notebook. XGBoost must be installed before importing it.